# Stage 4: Baseline Models — Random Forest & SVM

- Load cached log-Mel features
- Train RF and SVM
- Cross-validation with official US8K folds
- Evaluate on Test fold 10

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report

from features import load_feature_cache
from models.baseline import train_baseline, load_baseline, flatten
from config import TRAIN_FOLDS, TEST_FOLD, IDX_TO_CLASS, NUM_CLASSES, PLOTS_DIR

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#0d0d1a',
    'text.color': 'white', 'axes.labelcolor': '#aaa',
    'xtick.color': '#aaa', 'ytick.color': '#aaa',
})

X, y, folds = load_feature_cache('logmel')
print(f'Features: {X.shape}  Labels: {y.shape}')

## 1 · Train Baseline Models

In [ ]:
rf_model  = train_baseline('rf',  X, y, folds)
svm_model = train_baseline('svm', X, y, folds)
print('Both baseline models trained!')

## 2 · Fold-wise Cross-Validation

In [ ]:
from models.baseline import build_rf, build_svm

def fold_cv(model_fn, X, y, folds, name):
    accs = []
    all_folds = np.unique(folds)
    # Use folds 1-9 only (fold 10 = held-out test)
    cv_folds = [f for f in all_folds if f != TEST_FOLD]
    for val_fold in cv_folds:
        train_mask = (folds != val_fold) & (folds != TEST_FOLD)
        val_mask   = folds == val_fold
        X_flat = flatten(X)
        model = model_fn()
        model.fit(X_flat[train_mask], y[train_mask])
        pred = model.predict(X_flat[val_mask])
        acc  = accuracy_score(y[val_mask], pred)
        accs.append(acc)
        print(f'  {name} | Fold {val_fold:2d} → val_acc = {acc:.4f}')
    print(f'  {name} | Mean ± Std: {np.mean(accs):.4f} ± {np.std(accs):.4f}\n')
    return accs

print('=== Random Forest ===')
rf_accs = fold_cv(build_rf, X, y, folds, 'RF')
print('=== SVM ===')
svm_accs = fold_cv(build_svm, X, y, folds, 'SVM')

## 3 · CV Results Plot

In [ ]:
fold_labels = [str(f) for f in range(1, 10)]
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(fold_labels, [a*100 for a in rf_accs],  marker='o', lw=2, color='#00d4ff', label='Random Forest')
ax.plot(fold_labels, [a*100 for a in svm_accs], marker='s', lw=2, color='#ff6b6b', label='SVM')
ax.axhline(np.mean(rf_accs)*100,  linestyle='--', color='#00d4ff', alpha=0.5)
ax.axhline(np.mean(svm_accs)*100, linestyle='--', color='#ff6b6b', alpha=0.5)

ax.set_xlabel('Validation Fold')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Fold-wise Cross-Validation Accuracy', fontsize=13, fontweight='bold', color='white')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/baseline_cv.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 · Test Set Evaluation

In [ ]:
from evaluate import compute_metrics, plot_confusion_matrix

test_mask = folds == TEST_FOLD
X_test, y_test = X[test_mask], y[test_mask]
X_flat_test = flatten(X_test)

for name, model in [('RF', rf_model), ('SVM', svm_model)]:
    y_pred = model.predict(X_flat_test)
    compute_metrics(y_test, y_pred, name)
    plot_confusion_matrix(y_test, y_pred, name.lower())